In [4]:
## COLS TO KEEP WE ARE BUILDING

# LOCA2_FEATURES = [
#     'tmax_k', 'tmax_k_pop',
#     'tmin_k', 'tmin_k_pop', 
#     'tavg_k', 'trange_k',
#     'cdd65', 'cdd65_pop',
#     'cdd75', 'cdd75_pop',
#     'hdd65', 'hdd65_pop',
#     'spfh_peak_kgkg_mean', 'spfh_peak_kgkg_pop',
#     'wind_peak_ms_pop',
# ]

In [3]:
## COLS TO DROP FOR WEATHER:
# DROP_COLS = [
#     'cloud_cover_pct_mean', 'cloud_cover_pct_pop',
#     'dpt_morning_k_mean', 'dpt_morning_k_pop',
#     'dpt_afternoon_k_mean', 'dpt_afternoon_k_pop',
#     'wind_low_ms_mean', 'wind_low_ms_pop',  # only have peak wind in LOCA2
# ]

In [71]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import s3fs
from shapely.geometry import Point

fs = s3fs.S3FileSystem(anon=True)

# verify one path loads
ds_test = xr.open_zarr(
    's3://cadcat/loca2/ucsd/access-cm2/ssp370/r1i1p1f1/day/tasmax/d03/',
    storage_options={'anon': True},
    consolidated=False
)
print(ds_test)

<xarray.Dataset> Size: 35GB
Dimensions:  (time: 31411, lat: 495, lon: 559)
Coordinates:
  * time     (time) datetime64[ns] 251kB 2015-01-01T12:00:00 ... 2100-12-31T1...
  * lat      (lat) float32 2kB 29.58 29.61 29.64 29.67 ... 44.95 44.98 45.02
  * lon      (lon) float32 2kB -128.4 -128.4 -128.4 ... -111.0 -111.0 -111.0
Data variables:
    tasmax   (time, lat, lon) float32 35GB dask.array<chunksize=(1952, 123, 139), meta=np.ndarray>
Attributes: (12/85)
    Conventions:                         CF-1.7 CMIP-6.2
    ID_loca_routines_module:             $Id: loca_routines_module.F90,v 1.13...
    SIOCRD_netCDF_Version:               1.0
    SOURCE_loca_routines_module:         $Source: /home6/dwpierc2/src/mine/lo...
    activity_id:                         CMIP
    bias_correction:                     downscaling via PresRat, Pierce et a...
    ...                                  ...
    title:                               LOCA statistically downscaled climat...
    tracking_id:         

In [72]:
### BUILD COUNTY MASK
# Pull the LOCA2 lat/lon grid
lats = ds_test.lat.values  # 1D (495,)
lons = ds_test.lon.values  # 1D (559,)

# Make 2D meshgrid
lon_2d, lat_2d = np.meshgrid(lons, lats)

# Load California counties
counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
).query("STATEFP == '06'").to_crs("EPSG:4326")

# Flatten to points
points_df = pd.DataFrame({
    "lat": lat_2d.flatten(),
    "lon": lon_2d.flatten()
})

gdf_points = gpd.GeoDataFrame(
    points_df,
    geometry=gpd.points_from_xy(points_df.lon, points_df.lat),
    crs="EPSG:4326"
)

# Spatial join
gdf_joined = gpd.sjoin(gdf_points, counties[["NAME", "geometry"]], how="left", predicate="within")

# Build 2D county mask
county_mask = (
    gdf_joined["NAME"]
    .fillna("Outside_CA")
    .to_numpy(dtype="object")
    .reshape(lat_2d.shape)
)

print(f"Mask shape: {county_mask.shape}")
print(f"CA counties found: {len(np.unique(county_mask[county_mask != 'Outside_CA']))}")
print(f"Sample: {np.unique(county_mask)[:5]}")

Mask shape: (495, 559)
CA counties found: 58
Sample: ['Alameda' 'Alpine' 'Amador' 'Butte' 'Calaveras']


In [73]:
pop = xr.open_zarr("processed/worldpop_population_2019_2021.zarr", consolidated=False)
pop_2020 = pop["population"].sel(year=2020)
print(pop_2020)
print("\nCoords:", dict(pop_2020.coords))

<xarray.DataArray 'population' (y: 535, x: 457)> Size: 978kB
dask.array<getitem, shape=(535, 457), dtype=float32, chunksize=(256, 256), chunktype=numpy.ndarray>
Coordinates:
    year     int32 4B 2020
Dimensions without coordinates: y, x
Attributes:
    aggregation:  nearest_urma_cell_center_sum
    source:       WorldPop R2025A 1km_ua constrained (CA clipped)
    units:        persons

Coords: {'year': <xarray.DataArray 'year' ()> Size: 4B
array(2020, dtype=int32)
Coordinates:
    year     int32 4B 2020}


In [75]:
# Fix URMA lon to -180/180
urma_lon_fixed = np.where(urma_lon > 180, urma_lon - 360, urma_lon)

print(f"URMA lon fixed range: {urma_lon_fixed.min():.2f} to {urma_lon_fixed.max():.2f}")

# Attach lat/lon to WorldPop as coordinates
pop_2020_coords = pop_2020.assign_coords(
    lat=(("y", "x"), urma_lat),
    lon=(("y", "x"), urma_lon_fixed)
).compute()

# Flatten WorldPop to a dataframe with lat/lon
pop_df = pd.DataFrame({
    "lat": urma_lat.flatten(),
    "lon": urma_lon_fixed.flatten(),
    "population": pop_2020_coords.values.flatten()
})

# Nearest neighbor lookup: for each LOCA2 grid point, find closest URMA/WorldPop point
from scipy.spatial import cKDTree

urma_points = np.column_stack([pop_df.lat, pop_df.lon])
loca2_points = np.column_stack([lat_2d.flatten(), lon_2d.flatten()])

tree = cKDTree(urma_points)
_, idx = tree.query(loca2_points)

pop_loca2 = pop_df.population.values[idx].reshape(lat_2d.shape)

print(f"WorldPop on LOCA2 grid shape: {pop_loca2.shape}")
print(f"Population range: {pop_loca2.min():.1f} to {pop_loca2.max():.1f}")
print(f"Total CA population check: {pop_loca2[county_mask != 'Outside_CA'].sum():,.0f}")

URMA lon fixed range: -127.48 to -112.20
WorldPop on LOCA2 grid shape: (495, 559)
Population range: 0.0 to 126042.1
Total CA population check: 26,984,744


In [76]:
# Load just the coords from any URMA zarr
ds_urma = xr.open_zarr("01_URMA_RAW/URMA_RAW_2023_full.zarr", consolidated=False)

urma_lat = ds_urma.latitude.values  # (535, 457)
urma_lon = ds_urma.longitude.values  # (535, 457)

print(f"URMA lat shape: {urma_lat.shape}")
print(f"URMA lat range: {urma_lat.min():.2f} to {urma_lat.max():.2f}")
print(f"URMA lon range: {urma_lon.min():.2f} to {urma_lon.max():.2f}")

URMA lat shape: (535, 457)
URMA lat range: 30.40 to 43.90
URMA lon range: 232.52 to 247.80


In [62]:
# ##SSP370
# def build_loca2_features(model, member, county_mask, pop_loca2):
#     """
#     Load 4 LOCA2 variables, aggregate to county level, build all features.
#     Returns a dataframe with (date, county, features)
#     """
#     base = f's3://cadcat/loca2/ucsd/{model}/ssp370/{member}/day'
#     opts = {'anon': True}

#     print(f"Loading {model} {member}...")
#     tasmax = xr.open_zarr(f'{base}/tasmax/d03/', storage_options=opts, consolidated=False)['tasmax']
#     tasmin = xr.open_zarr(f'{base}/tasmin/d03/', storage_options=opts, consolidated=False)['tasmin']
#     wspeed = xr.open_zarr(f'{base}/wspeed/d03/', storage_options=opts, consolidated=False)['wspeed']
#     huss   = xr.open_zarr(f'{base}/huss/d03/',   storage_options=opts, consolidated=False)['huss']

#     # flatten spatial dims for groupby
#     ca_mask = county_mask.flatten()
#     pop_flat = pop_loca2.flatten()
#     ca_idx = ca_mask != 'Outside_CA'

#     times = tasmax.time.values
#     results = []

#     print(f"Processing {len(times)} days...")
#     for i, t in enumerate(times):
#         if i % 1000 == 0:
#             print(f"  day {i}/{len(times)}")

#         tmax = tasmax.isel(time=i).values.flatten()
#         tmin = tasmin.isel(time=i).values.flatten()
#         ws   = wspeed.isel(time=i).values.flatten()
#         hu   = huss.isel(time=i).values.flatten()

#         tavg = (tmax + tmin) / 2
#         base_k = (65 - 32) * 5/9 + 273.15
#         base75_k = (75 - 32) * 5/9 + 273.15

#         df_grid = pd.DataFrame({
#             'county': ca_mask,
#             'population': pop_flat,
#             'tmax': tmax, 'tmin': tmin,
#             'tavg': tavg,
#             'wspeed': ws, 'huss': hu,
#             'cdd65': np.maximum(tavg - base_k, 0),
#             'hdd65': np.maximum(base_k - tavg, 0),
#             'cdd75': np.maximum(tavg - base75_k, 0),
#         })[ca_idx]

#         grp = df_grid.groupby('county')
#         pop_grp = grp['population']

#         def pw_mean(col):
#             return (df_grid[col] * df_grid['population']).groupby(df_grid['county']).sum() / pop_grp.sum()

#         row = pd.DataFrame({
#             'tmax_k':               grp['tmax'].mean(),
#             'tmax_k_pop':           pw_mean('tmax'),
#             'tmin_k':               grp['tmin'].mean(),
#             'tmin_k_pop':           pw_mean('tmin'),
#             'tavg_k':               grp['tavg'].mean(),
#             'trange_k':             grp['tmax'].mean() - grp['tmin'].mean(),
#             'cdd65':                grp['cdd65'].mean(),
#             'cdd65_pop':            pw_mean('cdd65'),
#             'cdd75':                grp['cdd75'].mean(),
#             'cdd75_pop':            pw_mean('cdd75'),
#             'hdd65':                grp['hdd65'].mean(),
#             'hdd65_pop':            pw_mean('hdd65'),
#             'spfh_peak_kgkg_mean':  grp['huss'].mean(),
#             'spfh_peak_kgkg_pop':   pw_mean('huss'),
#             'wind_peak_ms_pop':     pw_mean('wspeed'),
#         })
#         row['date'] = t
#         results.append(row.reset_index())

#     df = pd.concat(results).rename(columns={'county': 'county'})
#     df['model'] = model
#     df['member'] = member
#     return df

In [13]:
# # Test on just 3 days of access-cm2 r1i1p1f1
# base = 's3://cadcat/loca2/ucsd/access-cm2/ssp370/r1i1p1f1/day'
# opts = {'anon': True}

# tasmax = xr.open_zarr(f'{base}/tasmax/d03/', storage_options=opts, consolidated=False)['tasmax']
# tasmin = xr.open_zarr(f'{base}/tasmin/d03/', storage_options=opts, consolidated=False)['tasmin']
# wspeed = xr.open_zarr(f'{base}/wspeed/d03/', storage_options=opts, consolidated=False)['wspeed']
# huss   = xr.open_zarr(f'{base}/huss/d03/',   storage_options=opts, consolidated=False)['huss']

# ca_mask = county_mask.flatten()
# pop_flat = pop_loca2.flatten()
# ca_idx = ca_mask != 'Outside_CA'
# base_k = (65 - 32) * 5/9 + 273.15
# base75_k = (75 - 32) * 5/9 + 273.15

# results = []
# for i in range(3):
#     tmax = tasmax.isel(time=i).values.flatten()
#     tmin = tasmin.isel(time=i).values.flatten()
#     ws   = wspeed.isel(time=i).values.flatten()
#     hu   = huss.isel(time=i).values.flatten()
#     tavg = (tmax + tmin) / 2

#     df_grid = pd.DataFrame({
#         'county': ca_mask, 'population': pop_flat,
#         'tmax': tmax, 'tmin': tmin, 'tavg': tavg,
#         'wspeed': ws, 'huss': hu,
#         'cdd65': np.maximum(tavg - base_k, 0),
#         'hdd65': np.maximum(base_k - tavg, 0),
#         'cdd75': np.maximum(tavg - base75_k, 0),
#     })[ca_idx]

#     grp = df_grid.groupby('county')
#     def pw_mean(col):
#         return (df_grid[col] * df_grid['population']).groupby(df_grid['county']).sum() / grp['population'].sum()

#     row = pd.DataFrame({
#         'tmax_k': grp['tmax'].mean(), 'tmax_k_pop': pw_mean('tmax'),
#         'tmin_k': grp['tmin'].mean(), 'tmin_k_pop': pw_mean('tmin'),
#         'tavg_k': grp['tavg'].mean(), 'trange_k': grp['tmax'].mean() - grp['tmin'].mean(),
#         'cdd65': grp['cdd65'].mean(), 'cdd65_pop': pw_mean('cdd65'),
#         'cdd75': grp['cdd75'].mean(), 'cdd75_pop': pw_mean('cdd75'),
#         'hdd65': grp['hdd65'].mean(), 'hdd65_pop': pw_mean('hdd65'),
#         'spfh_peak_kgkg_mean': grp['huss'].mean(), 'spfh_peak_kgkg_pop': pw_mean('huss'),
#         'wind_peak_ms_pop': pw_mean('wspeed'),
#     })
#     row['date'] = tasmax.time.values[i]
#     results.append(row.reset_index())

# test_df = pd.concat(results)
# print(test_df.shape)
# print(test_df.head())
# print("\nColumns:", test_df.columns.tolist())
# print("\nAny NaNs:", test_df.isnull().sum().sum())

(174, 17)
      county      tmax_k  tmax_k_pop      tmin_k  tmin_k_pop      tavg_k  \
0    Alameda  288.120911  279.002045  282.864258  273.939484  285.492615   
1     Alpine  277.911682  281.104187  267.476379  270.884552  272.694061   
2     Amador  284.854980  286.644531  276.764709  278.209167  280.809845   
3      Butte  286.498383  288.885590  280.688995  282.471863  283.593689   
4  Calaveras  285.807129  287.008362  276.382111  277.595062  281.094604   

    trange_k  cdd65  cdd65_pop  cdd75  cdd75_pop      hdd65  hdd65_pop  \
0   5.256653    0.0        0.0    0.0        0.0   5.990736   4.534609   
1  10.435303    0.0        0.0    0.0        0.0  18.789295  15.488959   
2   8.090271    0.0        0.0    0.0        0.0  10.673494   9.056476   
3   5.809387    0.0        0.0    0.0        0.0   7.889657   5.804631   
4   9.425018    0.0        0.0    0.0        0.0  10.388729   9.181630   

   spfh_peak_kgkg_mean  spfh_peak_kgkg_pop  wind_peak_ms_pop  \
0             0.007602  

In [77]:
### TEST SINGLE DAY
import time, os

# Time a single day fetch
start = time.time()
_ = tasmax.isel(time=0).values
elapsed = time.time() - start
print(f"Single day load: {elapsed:.1f}s")
print(f"Estimated 31,411 days × 4 vars: {elapsed * 4 * 31411 / 3600:.1f} hrs per run")
print(f"Estimated 25 runs total: {elapsed * 4 * 31411 * 25 / 3600:.1f} hrs")

# Output size estimate
# 58 counties × 31,411 days × 15 features × 4 bytes (float32)
bytes_per_run = 58 * 31411 * 15 * 4
print(f"\nOutput size per run (parquet est): {bytes_per_run / 1e6:.1f} MB")
print(f"Output size 25 runs: {bytes_per_run * 25 / 1e6:.0f} MB")

# Check available disk
statvfs = os.statvfs('.')
free_gb = statvfs.f_frsize * statvfs.f_bavail / 1e9
print(f"\nFree disk space: {free_gb:.1f} GB")

Single day load: 18.4s
Estimated 31,411 days × 4 vars: 641.4 hrs per run
Estimated 25 runs total: 16034.8 hrs

Output size per run (parquet est): 109.3 MB
Output size 25 runs: 2733 MB

Free disk space: 81.1 GB


In [78]:
# Define globally so we can use in debugging and in the function
ca_mask_flat    = county_mask.flatten()
pop_flat_all    = pop_loca2.flatten()
ca_idx          = ca_mask_flat != 'Outside_CA'
ca_counties     = ca_mask_flat[ca_idx]
ca_pop          = pop_flat_all[ca_idx]
unique_counties = np.unique(ca_counties)

county_pixel_idx = {c: np.where(ca_counties == c)[0] for c in unique_counties}
county_pop       = {c: ca_pop[county_pixel_idx[c]] for c in unique_counties}

# Now check Alameda
cidx = county_pixel_idx['Alameda']
print(f"Alameda pixel count: {len(cidx)}")
print(f"Alameda raw values day 0: {raw[0].flatten()[ca_idx][cidx][:5]}")
print(f"Any NaN in Alameda?: {np.isnan(raw[0].flatten()[ca_idx][cidx]).any()}")

Alameda pixel count: 226
Alameda raw values day 0: [289.19693 289.49683 289.54468 290.46832 291.00018]
Any NaN in Alameda?: True


In [65]:
# # Debug - check raw values before any processing
# tmax_da = xr.open_zarr('s3://cadcat/loca2/ucsd/access-cm2/ssp370/r1i1p1f1/day/tasmax/d03/', 
#                         storage_options={'anon': True}, consolidated=False)['tasmax'].sel(time='2015')

# times = tmax_da.time.values
# raw = tmax_da.values  # (365, 495, 559)

# print(f"Raw shape: {raw.shape}")
# print(f"Raw NaN count: {np.isnan(raw).sum()}")
# print(f"Raw min/max: {np.nanmin(raw):.2f} / {np.nanmax(raw):.2f}")
# print(f"Raw sample [0, 100, 100]: {raw[0, 100, 100]}")

# # Check ca_idx is finding pixels
# print(f"\nca_idx True count: {ca_idx.sum()} (should be >0)")
# print(f"ca_counties sample: {ca_counties[:5]}")

# # Check a specific county
# cidx = county_pixel_idx['Alameda']
# print(f"\nAlameda pixel count: {len(cidx)}")
# print(f"Alameda raw values day 0: {raw[0].flatten()[ca_idx][cidx][:5]}")

Raw shape: (365, 495, 559)
Raw NaN count: 69612800
Raw min/max: 249.78 / 327.23
Raw sample [0, 100, 100]: nan

ca_idx True count: 44116 (should be >0)
ca_counties sample: ['San Diego' 'San Diego' 'San Diego' 'San Diego' 'San Diego']

Alameda pixel count: 226
Alameda raw values day 0: [289.19693 289.49683 289.54468 290.46832 291.00018]


In [43]:
base_k   = (65 - 32) * 5/9 + 273.15
base75_k = (75 - 32) * 5/9 + 273.15

def process_one_run(model, member, years=range(2015, 2041)):
    base = f's3://cadcat/loca2/ucsd/{model}/ssp370/{member}/day'
    opts = {'anon': True}
    all_years = []

    for year in years:
        start = time.time()
        tmax_da = xr.open_zarr(f'{base}/tasmax/d03/', storage_options=opts, consolidated=False)['tasmax'].sel(time=str(year))
        tmin_da = xr.open_zarr(f'{base}/tasmin/d03/', storage_options=opts, consolidated=False)['tasmin'].sel(time=str(year))
        ws_da   = xr.open_zarr(f'{base}/wspeed/d03/', storage_options=opts, consolidated=False)['wspeed'].sel(time=str(year))
        hu_da   = xr.open_zarr(f'{base}/huss/d03/',   storage_options=opts, consolidated=False)['huss'].sel(time=str(year))

        times = tmax_da.time.values
        tmax = tmax_da.values.reshape(len(times), -1)[:, ca_idx]
        tmin = tmin_da.values.reshape(len(times), -1)[:, ca_idx]
        ws   = ws_da.values.reshape(len(times),   -1)[:, ca_idx]
        hu   = hu_da.values.reshape(len(times),   -1)[:, ca_idx]
        tavg = (tmax + tmin) / 2

        rows = []
        for cname in unique_counties:
            cidx = county_pixel_idx[cname]
            cpop = county_pop[cname]

            tm = tmax[:, cidx]
            tn = tmin[:, cidx]
            tv = tavg[:, cidx]
            w  = ws[:,   cidx]
            h  = hu[:,   cidx]

            def smean(arr): 
                return np.nanmean(arr, axis=1)

            def pmean(arr):
                # mask population where data is NaN
                valid = ~np.isnan(arr)
                weighted = np.where(valid, arr * cpop, 0).sum(axis=1)
                pop_sum  = np.where(valid, cpop, 0).sum(axis=1)
                return np.where(pop_sum > 0, weighted / pop_sum, np.nan)

            rows.append(pd.DataFrame({
                'date':                 times,
                'county':               cname,
                'tmax_k':               smean(tm),
                'tmax_k_pop':           pmean(tm),
                'tmin_k':               smean(tn),
                'tmin_k_pop':           pmean(tn),
                'tavg_k':               smean(tv),
                'trange_k':             smean(tm) - smean(tn),
                'cdd65':                np.nanmean(np.maximum(tv - base_k,   0), axis=1),
                'cdd65_pop':            pmean(np.maximum(tv - base_k,   0)),
                'cdd75':                np.nanmean(np.maximum(tv - base75_k, 0), axis=1),
                'cdd75_pop':            pmean(np.maximum(tv - base75_k, 0)),
                'hdd65':                np.nanmean(np.maximum(base_k - tv,   0), axis=1),
                'hdd65_pop':            pmean(np.maximum(base_k - tv,   0)),
                'spfh_peak_kgkg_mean':  smean(h),
                'spfh_peak_kgkg_pop':   pmean(h),
                'wind_peak_ms_pop':     pmean(w),
            }))

        year_df = pd.concat(rows)
        all_years.append(year_df)
        print(f"  {year} done in {time.time()-start:.1f}s — shape {year_df.shape} — NaNs: {year_df.isnull().sum().sum()}")

    return pd.concat(all_years).reset_index(drop=True)

print("Testing 2015...")
test = process_one_run('access-cm2', 'r1i1p1f1', years=[2015])
print(test.shape)
print(test.head())
print("NaNs:", test.isnull().sum().sum())

Testing 2015...
  2015 done in 82.6s — shape (21170, 17) — NaNs: 0
(21170, 17)
                 date   county      tmax_k  tmax_k_pop      tmin_k  \
0 2015-01-01 12:00:00  Alameda  288.120819  289.405334  282.864258   
1 2015-01-02 12:00:00  Alameda  288.885742  289.794983  281.713898   
2 2015-01-03 12:00:00  Alameda  288.761902  289.770355  283.798615   
3 2015-01-04 12:00:00  Alameda  288.225800  289.308685  282.914337   
4 2015-01-05 12:00:00  Alameda  291.417236  292.448334  285.845306   

   tmin_k_pop      tavg_k  trange_k  cdd65  cdd65_pop  cdd75  cdd75_pop  \
0  284.153961  285.492554  5.256561    0.0        0.0    0.0        0.0   
1  283.616241  285.299988  7.171844    0.0        0.0    0.0        0.0   
2  285.142456  286.280273  4.963287    0.0        0.0    0.0        0.0   
3  284.002014  285.570099  5.311462    0.0        0.0    0.0        0.0   
4  286.748657  288.631378  5.571930    0.0        0.0    0.0        0.0   

      hdd65  hdd65_pop  spfh_peak_kgkg_mean  spfh

In [28]:
### RUN FOR SSP370
##import os

# OUTPUT_DIR = '04_LOCA2_FEATURES'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# CLEAN_ENSEMBLE = {
#     'access-cm2':    ['r1i1p1f1'],
#     'ec-earth3':     ['r1i1p1f1', 'r4i1p1f1'],
#     'ec-earth3-veg': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1'],
#     'gfdl-esm4':     ['r1i1p1f1'],
#     'inm-cm5-0':     ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
#     'mpi-esm1-2-hr': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r10i1p1f1'],
#     'mri-esm2-0':    ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
# }

# total_runs = sum(len(v) for v in CLEAN_ENSEMBLE.values())
# completed = 0

# for model, members in CLEAN_ENSEMBLE.items():
#     for member in members:
#         out_path = f'{OUTPUT_DIR}/{model}_{member}.parquet'
        
#         if os.path.exists(out_path):
#             print(f"SKIP {model} {member} — already exists")
#             completed += 1
#             continue
        
#         print(f"\n[{completed+1}/{total_runs}] {model} {member}")
#         try:
#             df = process_one_run(model, member, years=range(2015, 2041))
#             df['model'] = model
#             df['member'] = member
#             df.to_parquet(out_path, index=False)
#             print(f"  Saved to {out_path} — {df.shape}")
#         except Exception as e:
#             print(f"  ERROR: {e}")
        
#         completed += 1

# print("\nDone!")

SKIP access-cm2 r1i1p1f1 — already exists
SKIP ec-earth3 r1i1p1f1 — already exists
SKIP ec-earth3 r4i1p1f1 — already exists
SKIP ec-earth3-veg r1i1p1f1 — already exists
SKIP ec-earth3-veg r2i1p1f1 — already exists
SKIP ec-earth3-veg r3i1p1f1 — already exists
SKIP ec-earth3-veg r4i1p1f1 — already exists
SKIP gfdl-esm4 r1i1p1f1 — already exists
SKIP inm-cm5-0 r1i1p1f1 — already exists
SKIP inm-cm5-0 r2i1p1f1 — already exists
SKIP inm-cm5-0 r3i1p1f1 — already exists

[12/24] inm-cm5-0 r4i1p1f1
  2015 done in 97.9s — shape (21170, 17) — NaNs: 0
  2016 done in 105.2s — shape (21228, 17) — NaNs: 0
  2017 done in 65.8s — shape (21170, 17) — NaNs: 0
  2018 done in 63.1s — shape (21170, 17) — NaNs: 0
  2019 done in 58.2s — shape (21170, 17) — NaNs: 0
  2020 done in 119.5s — shape (21228, 17) — NaNs: 0
  2021 done in 73.1s — shape (21170, 17) — NaNs: 0
  2022 done in 94.4s — shape (21170, 17) — NaNs: 0
  2023 done in 108.6s — shape (21170, 17) — NaNs: 0
  2024 done in 90.8s — shape (21228, 17) —

In [29]:
## FOR SSP370, SEE FILES
# import os
# import glob

# files = glob.glob('04_LOCA2_FEATURES/*.parquet')
# print(f"Completed: {len(files)}/25")
# for f in sorted(files):
#     size_mb = os.path.getsize(f) / 1e6
#     print(f"  {os.path.basename(f)}: {size_mb:.1f} MB")

Completed: 25/25
  access-cm2_ssp370_r1i1p1f1.parquet: 35.7 MB
  ec-earth3-veg_ssp370_r1i1p1f1.parquet: 35.7 MB
  ec-earth3-veg_ssp370_r2i1p1f1.parquet: 35.7 MB
  ec-earth3-veg_ssp370_r3i1p1f1.parquet: 35.5 MB
  ec-earth3-veg_ssp370_r4i1p1f1.parquet: 35.7 MB
  ec-earth3_ssp370_r1i1p1f1.parquet: 35.8 MB
  ec-earth3_ssp370_r4i1p1f1.parquet: 35.7 MB
  gfdl-esm4_ssp370_r1i1p1f1.parquet: 35.6 MB
  inm-cm5-0_r5i1p1f1.parquet: 35.6 MB
  inm-cm5-0_ssp370_r1i1p1f1.parquet: 35.6 MB
  inm-cm5-0_ssp370_r2i1p1f1.parquet: 35.5 MB
  inm-cm5-0_ssp370_r3i1p1f1.parquet: 35.5 MB
  inm-cm5-0_ssp370_r4i1p1f1.parquet: 35.5 MB
  mpi-esm1-2-hr_r10i1p1f1.parquet: 35.6 MB
  mpi-esm1-2-hr_r1i1p1f1.parquet: 35.5 MB
  mpi-esm1-2-hr_r2i1p1f1.parquet: 35.7 MB
  mpi-esm1-2-hr_r3i1p1f1.parquet: 35.6 MB
  mpi-esm1-2-hr_r4i1p1f1.parquet: 35.5 MB
  mpi-esm1-2-hr_r5i1p1f1.parquet: 35.5 MB
  mri-esm2-0_r1i1p1f1.parquet: 35.7 MB
  mri-esm2-0_r2i1p1f1.parquet: 35.6 MB
  mri-esm2-0_r3i1p1f1.parquet: 35.7 MB
  mri-esm2-0_r4i1p

# SSP245

In [82]:
import s3fs
fs = s3fs.S3FileSystem(anon=True)
fs.ls('s3://cadcat/loca2/ucsd/mri-esm2-0/ssp245/r1i1p1f1/')

['cadcat/loca2/ucsd/mri-esm2-0/ssp245/r1i1p1f1/day',
 'cadcat/loca2/ucsd/mri-esm2-0/ssp245/r1i1p1f1/mon',
 'cadcat/loca2/ucsd/mri-esm2-0/ssp245/r1i1p1f1/yrmax']

In [83]:
import s3fs

fs = s3fs.S3FileSystem(anon=True)

# Get all models
models = fs.ls('s3://cadcat/loca2/ucsd/')
models = [m.split('/')[-1] for m in models]

# Check which have all 4 variables under ssp245
required_vars = {'tasmax', 'tasmin', 'wspeed', 'huss'}
good_models = []

for model in models:
    try:
        # Try r1i1p1f1 first, most common ensemble member
        base = f's3://cadcat/loca2/ucsd/{model}/ssp245/r1i1p1f1'
        contents = fs.ls(base)
        available = {c.split('/')[-1] for c in contents}
        if required_vars.issubset(available):
            good_models.append(model)
            print(f"✓ {model}")
        else:
            missing = required_vars - available
            print(f"✗ {model} — missing: {missing}")
    except Exception as e:
        print(f"✗ {model} — error: {e}")

print(f"\nModels with all variables: {good_models}")

✗ access-cm2 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ cesm2-lens — error: cadcat/loca2/ucsd/cesm2-lens/ssp245/r1i1p1f1
✗ cnrm-esm2-1 — error: cadcat/loca2/ucsd/cnrm-esm2-1/ssp245/r1i1p1f1
✗ ec-earth3 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ ec-earth3-veg — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ fgoals-g3 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ gfdl-esm4 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ hadgem3-gc31-ll — error: cadcat/loca2/ucsd/hadgem3-gc31-ll/ssp245/r1i1p1f1
✗ inm-cm5-0 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ ipsl-cm6a-lr — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ kace-1-0-g — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ miroc6 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ mpi-esm1-2-hr — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ mri-esm2-0 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}
✗ taiesm1 — missing: {'tasmin', 'tasmax', 'huss', 'wspeed'}

Models with all variables: []


In [91]:
base_k   = (65 - 32) * 5/9 + 273.15
base75_k = (75 - 32) * 5/9 + 273.15

def process_one_run(model, member, years=range(2015, 2041), scenario='ssp245'):
    base = f's3://cadcat/loca2/ucsd/{model}/{scenario}/{member}/day'
    opts = {'anon': True}
    all_years = []
    for year in years:
        start = time.time()
        tmax_da = xr.open_zarr(f'{base}/tasmax/d03/', storage_options=opts, consolidated=False)['tasmax'].sel(time=str(year))
        tmin_da = xr.open_zarr(f'{base}/tasmin/d03/', storage_options=opts, consolidated=False)['tasmin'].sel(time=str(year))
        ws_da   = xr.open_zarr(f'{base}/wspeed/d03/', storage_options=opts, consolidated=False)['wspeed'].sel(time=str(year))
        hu_da   = xr.open_zarr(f'{base}/huss/d03/',   storage_options=opts, consolidated=False)['huss'].sel(time=str(year))
        times = tmax_da.time.values
        tmax = tmax_da.values.reshape(len(times), -1)[:, ca_idx]
        tmin = tmin_da.values.reshape(len(times), -1)[:, ca_idx]
        ws   = ws_da.values.reshape(len(times),   -1)[:, ca_idx]
        hu   = hu_da.values.reshape(len(times),   -1)[:, ca_idx]
        tavg = (tmax + tmin) / 2
        rows = []
        for cname in unique_counties:
            cidx = county_pixel_idx[cname]
            cpop = county_pop[cname]
            tm = tmax[:, cidx]
            tn = tmin[:, cidx]
            tv = tavg[:, cidx]
            w  = ws[:,   cidx]
            h  = hu[:,   cidx]
            def smean(arr):
                return np.nanmean(arr, axis=1)
            def pmean(arr):
                valid   = ~np.isnan(arr)
                weighted = np.where(valid, arr * cpop, 0).sum(axis=1)
                pop_sum  = np.where(valid, cpop, 0).sum(axis=1)
                return np.where(pop_sum > 0, weighted / pop_sum, np.nan)
            rows.append(pd.DataFrame({
                'date':                 times,
                'county':               cname,
                'tmax_k':               smean(tm),
                'tmax_k_pop':           pmean(tm),
                'tmin_k':               smean(tn),
                'tmin_k_pop':           pmean(tn),
                'tavg_k':               smean(tv),
                'trange_k':             smean(tm) - smean(tn),
                'cdd65':                np.nanmean(np.maximum(tv - base_k,   0), axis=1),
                'cdd65_pop':            pmean(np.maximum(tv - base_k,   0)),
                'cdd75':                np.nanmean(np.maximum(tv - base75_k, 0), axis=1),
                'cdd75_pop':            pmean(np.maximum(tv - base75_k, 0)),
                'hdd65':                np.nanmean(np.maximum(base_k - tv,   0), axis=1),
                'hdd65_pop':            pmean(np.maximum(base_k - tv,   0)),
                'spfh_peak_kgkg_mean':  smean(h),
                'spfh_peak_kgkg_pop':   pmean(h),
                'wind_peak_ms_pop':     pmean(w),
            }))
        year_df = pd.concat(rows)
        all_years.append(year_df)
        print(f"  {year} done in {time.time()-start:.1f}s — shape {year_df.shape} — NaNs: {year_df.isnull().sum().sum()}")
    return pd.concat(all_years).reset_index(drop=True)

# # Test
# print("Testing 2015...")
# test = process_one_run('access-cm2', 'r1i1p1f1', years=[2015], scenario='ssp245')
# print(test.shape)
# print(test.head())
# print("NaNs:", test.isnull().sum().sum())

In [99]:
import os
import glob

files = glob.glob('04_ssp245_LOCA2_FEAT/*.parquet')
print(f"Completed: {len(files)}")
for f in sorted(files):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f)}: {size_mb:.1f} MB")

Completed: 15
  access-cm2_ssp245_r1i1p1f1.parquet: 35.7 MB
  cnrm-esm2-1_ssp245_r1i1p1f2.parquet: 35.6 MB
  ec-earth3-veg_ssp245_r1i1p1f1.parquet: 35.8 MB
  ec-earth3-veg_ssp245_r2i1p1f1.parquet: 35.9 MB
  ec-earth3-veg_ssp245_r3i1p1f1.parquet: 35.7 MB
  ec-earth3-veg_ssp245_r4i1p1f1.parquet: 35.9 MB
  ec-earth3-veg_ssp245_r5i1p1f1.parquet: 35.7 MB
  ec-earth3_ssp245_r1i1p1f1.parquet: 35.7 MB
  ec-earth3_ssp245_r2i1p1f1.parquet: 35.8 MB
  ec-earth3_ssp245_r4i1p1f1.parquet: 35.6 MB
  gfdl-esm4_ssp245_r1i1p1f1.parquet: 35.6 MB
  inm-cm5-0_ssp245_r1i1p1f1.parquet: 35.6 MB
  mpi-esm1-2-hr_ssp245_r1i1p1f1.parquet: 35.6 MB
  mpi-esm1-2-hr_ssp245_r2i1p1f1.parquet: 35.6 MB
  mri-esm2-0_ssp245_r1i1p1f1.parquet: 35.7 MB


In [94]:
CLEAN_ENSEMBLE_SSP245 = {
    'access-cm2':    ['r1i1p1f1'],
    'cnrm-esm2-1':   ['r1i1p1f2'],
    'ec-earth3':     ['r1i1p1f1', 'r2i1p1f1', 'r4i1p1f1'],
    'ec-earth3-veg': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
    'gfdl-esm4':     ['r1i1p1f1'],
    'inm-cm5-0':     ['r1i1p1f1'],
    'mpi-esm1-2-hr': ['r1i1p1f1', 'r2i1p1f1'],
    'mri-esm2-0':    ['r1i1p1f1'],
}

In [98]:
CLEAN_ENSEMBLE_SSP245 = {
    'access-cm2':    ['r1i1p1f1'],
    'cnrm-esm2-1':   ['r1i1p1f2'],
    'ec-earth3':     ['r1i1p1f1', 'r2i1p1f1', 'r4i1p1f1'],
    'ec-earth3-veg': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
    'gfdl-esm4':     ['r1i1p1f1'],
    'inm-cm5-0':     ['r1i1p1f1'],
    'mpi-esm1-2-hr': ['r1i1p1f1', 'r2i1p1f1'],
    'mri-esm2-0':    ['r1i1p1f1'],
}

SCENARIO   = 'ssp245'
OUTPUT_DIR = '04_ssp245_LOCA2_FEAT'
os.makedirs(OUTPUT_DIR, exist_ok=True)

total_runs = sum(len(v) for v in CLEAN_ENSEMBLE_SSP245.values())
completed  = 0

for model, members in CLEAN_ENSEMBLE_SSP245.items():
    for member in members:
        out_path = f'{OUTPUT_DIR}/{model}_{SCENARIO}_{member}.parquet'
        if os.path.exists(out_path):
            print(f"SKIP {model} {SCENARIO} {member} — already exists")
            completed += 1
            continue
        print(f"\n[{completed+1}/{total_runs}] {model} {SCENARIO} {member}")
        try:
            df = process_one_run(model, member, years=range(2015, 2041), scenario=SCENARIO)
            df['model']    = model
            df['scenario'] = SCENARIO
            df['member']   = member
            df.to_parquet(out_path, index=False)
            print(f"  Saved → {out_path} — {df.shape}")
        except Exception as e:
            print(f"  ERROR: {e}")
        completed += 1

print("\nDone!")

SKIP access-cm2 ssp245 r1i1p1f1 — already exists

[2/15] cnrm-esm2-1 ssp245 r1i1p1f2
  2015 done in 78.5s — shape (21170, 17) — NaNs: 0
  2016 done in 67.8s — shape (21228, 17) — NaNs: 0
  2017 done in 87.6s — shape (21170, 17) — NaNs: 0
  2018 done in 86.8s — shape (21170, 17) — NaNs: 0
  2019 done in 96.7s — shape (21170, 17) — NaNs: 0
  2020 done in 121.8s — shape (21228, 17) — NaNs: 0
  2021 done in 102.4s — shape (21170, 17) — NaNs: 0
  2022 done in 117.2s — shape (21170, 17) — NaNs: 0
  2023 done in 102.4s — shape (21170, 17) — NaNs: 0
  2024 done in 108.6s — shape (21228, 17) — NaNs: 0
  2025 done in 122.1s — shape (21170, 17) — NaNs: 0
  2026 done in 112.4s — shape (21170, 17) — NaNs: 0
  2027 done in 100.9s — shape (21170, 17) — NaNs: 0
  2028 done in 104.2s — shape (21228, 17) — NaNs: 0
  2029 done in 112.3s — shape (21170, 17) — NaNs: 0
  2030 done in 100.9s — shape (21170, 17) — NaNs: 0
  2031 done in 146.7s — shape (21170, 17) — NaNs: 0
  2032 done in 109.3s — shape (21228